
# Project Report: Classification of "Wigglies" Data

**Team:** Sara Prasla (10001841), Richard Van Winkle (3214183)

**Date:** July 9, 2025

### 1. Introduction

The goal of this project is to develop a high-performing deep learning model to classify the "wigglies" data and compete in the associated Kaggle competition. The core task involves implementing, training, evaluating, and comparing at least two distinct neural network architectures: a Multi-Layer Perceptron (MLP) and a Convolutional Neural Network (CNN)..

We were provided with three datasets:
1.  `trainset.pth`: An annotated dataset used for training and validating our models.
2.  `testset_noLabels.pth`: An unannotated dataset containing the "wigglies" data that our final model must classify for the competition submission.

The primary challenge is to determine the most effective architecture for this specific dataset and to optimize its performance to achieve the highest possible accuracy. Our strategy focuses on a systematic comparison between a baseline MLP, which treats the data as a simple vector, and a more advanced CNN, which is designed to capture potential sequential or spatial patterns within the data. This report details our step-by-step methodology for model development, the training and evaluation process, and the justification for our final model choice.

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import pandas as pd
import os

#### 2. Data Preparation

Our first step is to load the data from the `.pth` files. We created two utility functions:
* `get_dataloaders`: Loads the `trainset.pth` file, which contains both data and labels. It then splits this data into a training set (80%) and a validation set (20%). The validation set is crucial for monitoring model performance on unseen data to prevent overfitting.
* `get_test_loader`: Loads the `testset_noLabels.pth` file for generating the final predictions for the Kaggle submission.

The `weights_only=False` argument is necessary because our data files contain `TensorDataset` objects, not just model weights.

In [ ]:
## Cell 2: Helper Functions (from utils.py)

"""
Loads and splits the training data into training and validation dataloaders.
Add weights_only=False because the file contains a TensorDataset object
"""
def get_dataloaders(train_path='../data/trainset.pth', batch_size=64, valid_split=0.2):
    train_data = torch.load(train_path, weights_only=False) 

    total_size = len(train_data)
    val_size = int(total_size * valid_split)
    train_size = total_size - val_size

    train_ds, val_ds = random_split(train_data, [train_size, val_size])

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader


"""
Loads the test data.
Also add weights_only=False here for the same reason
"""
def get_test_loader(test_path='../data/testset_noLabels.pth', batch_size=64):
    
    test_ds = torch.load(test_path, weights_only=False) 
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return test_loader


#### 3. Model Architectures and Training

##### Model 1: Multi-Layer Perceptron (MLP)

Our first model is a standard Feed-Forward Neural Network, or MLP. It treats the 40 input features as a flat vector.

**Architecture:**
* It consists of two hidden layers with 128 and 64 neurons, respectively.
* We use the ReLU activation function to introduce non-linearity.
* To combat overfitting, we added Dropout layers with a probability of 0.25 and weight_decay to the optimizer. These techniques force the model to learn more robust features.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim=40, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.net(x)

**Training:**
* We train the model using the Adam optimizer and CrossEntropyLoss function.
* To ensure we select the best version of the model, we implement **Early Stopping**. The training process automatically terminates if the validation accuracy does not improve for 20 consecutive epochs. The model with the highest validation accuracy is saved.
* We also use a ReduceLROnPlateau learning rate scheduler, which reduces the learning rate if the validation loss plateaus, allowing for more fine-grained convergence.


In [ ]:
# --- Configuration ---
save_path = "../outputs/best_mlp.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Hyperparameters ---
lr = 1e-3
epochs = 500
batch_size = 64
weight_decay = 1e-4         # L2 Regularisation - prevent overfitting by penalising large weights
early_stopping_patience = 25

# --- Setup ---
model = MLP(input_dim=40, num_classes=10).to(device)
train_loader, val_loader = get_dataloaders(batch_size=batch_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)                                # update model's weights to reduce the loss.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10)      # automatically reduce the learning rate if the model's performance stops improving

# --- Training Loop with Adam optimiser and Cross Entropy Loss---
epochs_no_improve = 0
best_val_acc = 0.0

for epoch in range(epochs):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()      #  clear out gradient so fresh calculation is done
        out = model(xb)            #  performs the forward pass, getting the model's raw output scores (logits) for the input batch xb
        loss = criterion(out, yb)  #  loss (error) for the current batch by comparing the model's predictions (out) with the true answers (yb)
        loss.backward()            #  backpropagation step, calculate gradient of the loss w/ respect to model parameter(weight and bias)
        optimizer.step()           #  uses gradient to reduce loss by updating model's weight
        
        train_loss += loss.item() * xb.size(0)   #  total loss for the epoch, not just the average
        _, pred = out.max(1)                     #  model's actual prediction
        train_correct += pred.eq(yb).sum().item()    #  counts the number of correct predictions in the batch
        train_total += yb.size(0)                   #  total of how many samples have been processed in the epoch so far
    
    train_loss /= train_total               #  final average training loss for each epoch
    train_acc = train_correct / train_total   #  final training accuracy

    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            
            val_loss += loss.item() * xb.size(0)
            _, pred = out.max(1)
            val_correct += pred.eq(yb).sum().item()
            val_total += yb.size(0)
    
    val_loss /= val_total
    val_acc = val_correct / val_total
    
    scheduler.step(val_loss)      # Stepping the scheduler ensures that we only reduce the learning rate when the model has truly stopped getting better at solving new problems.

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), save_path)
        print(f"-> New best model saved with Val Acc: {best_val_acc:.4f}")
    else:
        epochs_no_improve += 1

    if epochs_no_improve == early_stopping_patience:
        print("Early stopping triggered!")
        break

Using device: cuda
Epoch 1/500 | Train Loss: 2.2730 Acc: 0.1466 | Val Loss: 2.1705 Acc: 0.2250
-> New best model saved with Val Acc: 0.2250
Epoch 2/500 | Train Loss: 2.0820 Acc: 0.2081 | Val Loss: 1.8987 Acc: 0.2188
Epoch 3/500 | Train Loss: 1.9170 Acc: 0.2172 | Val Loss: 1.8000 Acc: 0.2500
-> New best model saved with Val Acc: 0.2500
Epoch 4/500 | Train Loss: 1.8534 Acc: 0.2400 | Val Loss: 1.7634 Acc: 0.2938
-> New best model saved with Val Acc: 0.2938
Epoch 5/500 | Train Loss: 1.8133 Acc: 0.2572 | Val Loss: 1.7344 Acc: 0.2938
Epoch 6/500 | Train Loss: 1.8037 Acc: 0.2653 | Val Loss: 1.7138 Acc: 0.2913
Epoch 7/500 | Train Loss: 1.7745 Acc: 0.2831 | Val Loss: 1.6925 Acc: 0.2963
-> New best model saved with Val Acc: 0.2963
Epoch 8/500 | Train Loss: 1.7427 Acc: 0.2947 | Val Loss: 1.6700 Acc: 0.3050
-> New best model saved with Val Acc: 0.3050
Epoch 9/500 | Train Loss: 1.7242 Acc: 0.3019 | Val Loss: 1.6422 Acc: 0.3187
-> New best model saved with Val Acc: 0.3187
Epoch 10/500 | Train Loss: 

#### 4. Prediction and Final Submission

After training, we use the best-performing **MLP model** to generate predictions on the test dataset. The model is loaded from the checkpoint saved during training (i.e., the epoch where it achieved the highest validation accuracy).

For each test sample, the model outputs a probability distribution over the possible classes. We select the class with the highest predicted probability as the final label.

These predictions are then saved to a `.csv` file in the required submission format.


In [16]:
# --- Configuration ---
weights_path = "../outputs/best_mlp.pth"
output_csv = "../outputs/submission_mlp.csv"

# --- Load Model ---
# Re-initialize the model structure
model = MLP(input_dim=40, num_classes=10)
# Load the saved weights
model.load_state_dict(torch.load(weights_path, map_location=device))
model.to(device)
model.eval()

# --- Load Test Data ---
test_loader = get_test_loader()

# --- Generate Predictions ---
all_preds = []
with torch.no_grad():
    for xb in test_loader:
        xb = xb[0] if isinstance(xb, (tuple, list)) else xb
        xb = xb.to(device)
        outputs = model(xb)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)

# --- Create Submission File ---
assert len(all_preds) == 1000, f"Expected 1000 predictions, got {len(all_preds)}"
df = pd.DataFrame({"ID": range(1000), "Category": all_preds})
df.to_csv(output_csv, index=False)

print(f"\nSuccessfully saved predictions to {output_csv}")
print(df.head())


Successfully saved predictions to ../outputs/submission_mlp.csv
   ID  Category
0   0         2
1   1         8
2   2         9
3   3         9
4   4         8
